# Phase C: V1 rerun (fp32) + V2 stability + V3 confidence

**Two runtimes, in parallel:**
- **Runtime A, GPU (T4/L4):** run *Setup 1–5*, then **section A**.
- **Runtime B, CPU only** (Runtime → Change runtime type → CPU): run *Setup 1–5*, then **section B**.
- When both are done, run **section C** in either runtime.

Everything saves progress to Drive. **After a disconnect, run Setup again, then re-run the cell that was interrupted. It resumes and does not start over.** Every cell can be re-run safely.

## Setup 1: paths, Drive, version check

In [1]:
from pathlib import Path
CODE_DIR = Path("/content/drive/MyDrive/csg520_code/src")
OUT_DIR  = Path("/content/drive/MyDrive/csg520_outputs")
CKPT_DIR = OUT_DIR / "checkpoints"
LOCAL    = Path("/content/leaf_xai")

from google.colab import drive
drive.mount("/content/drive")

# 1) the code on Drive must be the new version
for f in ("explain.py", "views.py", "robustness.py", "confidence.py"):
    assert (CODE_DIR / f).exists(), f"{f} is missing from {CODE_DIR}: upload it"
src = (CODE_DIR / "explain.py").read_text()
assert "--check-accuracy" in src and "def set_context" in src and "occ_logp" in src, \
    "explain.py on Drive is the OLD version: upload the patched one from your local src folder"
print("code: new explain.py + views/robustness/confidence found")

# 2) the checkpoints and cached files the runs need
needed = ["checkpoints/a1_resnet50.pt", "checkpoints/a1_efficientnet_b0.pt", "checkpoints/a1_resnet50_scratch.pt",
          "checkpoints/a3_cnn.pt", "checkpoints/a4_vit.pt", "checkpoints/a2_rf.joblib",
          "features/a2_test_n400_s256.npz", "preds/a3_cnn_test_probs.npz", "preds/a4_vit_test_probs.npz",
          "split/splits.csv", "split/classes.json"]
missing = [f for f in needed if not (OUT_DIR / f).exists()]
for f in needed:
    print(f"  {f:<42}", "MISSING" if f in missing else "OK")
assert not missing, f"missing on Drive: {missing}"

Mounted at /content/drive
code: new explain.py + views/robustness/confidence found
  checkpoints/a1_resnet50.pt                 OK
  checkpoints/a1_efficientnet_b0.pt          OK
  checkpoints/a1_resnet50_scratch.pt         OK
  checkpoints/a3_cnn.pt                      OK
  checkpoints/a4_vit.pt                      OK
  checkpoints/a2_rf.joblib                   OK
  features/a2_test_n400_s256.npz             OK
  preds/a3_cnn_test_probs.npz                OK
  preds/a4_vit_test_probs.npz                OK
  split/splits.csv                           OK
  split/classes.json                         OK


## Setup 2: code to local disk, packages

In [2]:
import shutil, os, subprocess, re, json
shutil.copytree(CODE_DIR, LOCAL / "src", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc"))
!pip -q install lime "timm>=0.9" kagglehub
import torch
HAS_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| CPU cores:", os.cpu_count())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
GPU: NVIDIA L4 | CPU cores: 12


## Setup 3: PlantVillage colour images (models) + segmented variant (leaf masks only)

In [3]:
import kagglehub
try:
    from google.colab import userdata
    os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
    os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
except Exception:
    pass
ds_path = Path(kagglehub.dataset_download("abdallahalidev/plantvillage-dataset"))

def find_variant(name):
    return next(p for p in sorted(ds_path.rglob(name))
                if p.is_dir() and sum(1 for c in p.iterdir() if c.is_dir()) == 38)

color, MASK_ROOT = find_variant("color"), find_variant("segmented")
raw = LOCAL / "data" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    raw.symlink_to(color)
print("colour :", color)
print("masks  :", MASK_ROOT)

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
colour : /kaggle/input/plantvillage-dataset/plantvillage dataset/color
masks  : /kaggle/input/plantvillage-dataset/plantvillage dataset/segmented


## Setup 4: frozen split

In [4]:
import pandas as pd
for f in ("splits.csv", "classes.json"):
    shutil.copy(OUT_DIR / "split" / f, LOCAL / "data" / f)
counts = pd.read_csv(LOCAL / "data" / "splits.csv")["split"].value_counts().to_dict()
print(counts)
assert counts == {"train": 38013, "val": 8146, "test": 8146}

# Arguments shared by every command below. Data and split come from /content/leaf_xai/data (the defaults).
ARGS = f'--ckpt-dir "{CKPT_DIR}" --output-dir "{OUT_DIR}"'
%cd /content/leaf_xai/src

{'train': 38013, 'test': 8146, 'val': 8146}
/content/leaf_xai/src


## Setup 5: move the old fp16 V1 results out of the way (safe to re-run)
Any finished V1 arm without the new `occ_logp` metrics was made by the old code. It is moved to
`csg520_outputs/explain_fp16_firstpass/` (kept, not deleted), so the new runs start clean and later
**resume** after a disconnect instead of restarting. Arms produced by the new code, or still in progress, are left alone.

In [5]:
exp, old = OUT_DIR / "explain", OUT_DIR / "explain_fp16_firstpass"
for arm_dir in sorted(exp.glob("a*")):
    s = arm_dir / "summary.json"
    if s.exists() and "occ_logp_lift_kappa" not in json.loads(s.read_text()).get("metrics", {}):
        old.mkdir(exist_ok=True)
        dest = old / arm_dir.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(arm_dir), str(dest))
        print("archived old result:", arm_dir.name)
print("current V1 folders:", [p.name for p in sorted(exp.glob("a*"))])

current V1 folders: []


# A. Runtime A (GPU): A1 arms, A3, A4

**A1. Smoke test** (3 images, throwaway folder). It checks that the new code runs on the GPU and writes the new metrics.

In [6]:
assert HAS_GPU, "Section A needs a GPU runtime (Runtime → Change runtime type → T4/L4)"
!python explain.py --arms a3_cnn --limit 3 --fresh --ckpt-dir "{CKPT_DIR}" --output-dir /content/smoke --mask-root "{MASK_ROOT}"
m = json.loads(Path("/content/smoke/explain/a3_cnn/summary.json").read_text())["metrics"]
assert "occ_logp_lift_kappa" in m, "smoke run did not produce the new metrics: stop and check explain.py"
print("\nsmoke test OK")

[explain] sample: 380 images (10/class), LIME on 114 (A2: 2/class) | fill=mean mean RGB=(110.5, 120.0, 93.1)
[explain] mask index: 54306 segmented images
[a3_cnn] 3/3 images  (1.8s/img, ~0 min left)

=== V1 / a3_cnn ===  images=3  sample acc=1.000  mean leaf fraction (null)=0.710  masks={'gt': 3}
  occ_score            mean 0.850  median 0.918  95% CI [0.698, 0.933]  n=3
  occ_lift_ratio       mean 1.213  median 1.139  95% CI [0.994, 1.504]  n=3
  occ_lift_kappa       mean 0.463  median 0.579  95% CI [-0.015, 0.825]  n=3
  occ_logp_score       mean 0.850  median 0.919  95% CI [0.699, 0.933]  n=3
  occ_logp_lift_kappa  mean 0.464  median 0.582  95% CI [-0.014, 0.825]  n=3
  lime_score           mean 0.820  median 0.807  95% CI [0.779, 0.873]  n=3
  lime_lift_ratio      mean 1.164  median 1.109  95% CI [1.083, 1.301]  n=3
  lime_lift_kappa      mean 0.365  median 0.345  95% CI [0.257, 0.493]  n=3
  spearman_occ_lime    mean 0.351  median 0.341  95% CI [0.095, 0.618]  n=3
  lime_r2       

**A2. V1 rerun, fp32.** Five arms × 380 images, about 5–15 min per arm. Resumes after a disconnect.

In [10]:
assert HAS_GPU
!python explain.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit {ARGS} --mask-root "{MASK_ROOT}" 2>&1 | tee -a "{OUT_DIR}/explain_fp32_gpu_log.txt"

[explain] sample: 380 images (10/class), LIME on 114 (A2: 2/class) | fill=mean mean RGB=(110.5, 120.0, 93.1)
[explain] mask index: 54306 segmented images
[a1_resnet50] resuming: 380 occlusion / 114 LIME done

=== V1 / a1_resnet50 ===  images=380  sample acc=0.997  mean leaf fraction (null)=0.587  masks={'gt': 370, 'exg': 10}
  occ_score            mean 0.616  median 0.639  95% CI [0.596, 0.636]  n=380
  occ_lift_ratio       mean 1.063  median 1.062  95% CI [1.035, 1.092]  n=365
  occ_lift_kappa       mean 0.074  median 0.104  95% CI [0.039, 0.108]  n=365
  occ_logp_score       mean 0.616  median 0.640  95% CI [0.596, 0.636]  n=380
  occ_logp_lift_kappa  mean 0.075  median 0.105  95% CI [0.040, 0.108]  n=365
  lime_score           mean 0.750  median 0.780  95% CI [0.722, 0.777]  n=114
  lime_lift_ratio      mean 1.373  median 1.286  95% CI [1.305, 1.443]  n=111
  lime_lift_kappa      mean 0.403  median 0.463  95% CI [0.351, 0.449]  n=111
  spearman_occ_lime    mean -0.001  median 0.006 

**A3. V3 confidence.** A3 and A4 read their saved probabilities. The three A1 arms are scored once on the full test set (8,146 images) and saved to `preds/`.
Check that the macro-F1 in each printout matches the `metrics/*.json` value next to it.

In [8]:
assert HAS_GPU
!python confidence.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit {ARGS} 2>&1 | tee -a "{OUT_DIR}/confidence_gpu_log.txt"


[confidence] wrote /content/drive/MyDrive/csg520_outputs/preds/a1_resnet50_test_probs.npz

=== V3 / a1_resnet50 ===  n=8146  macro-F1=0.9963 (metrics/a1_resnet50.json: 0.9963)
  confidence  correct 0.871  wrong 0.609  gap 0.261 [0.148, 0.379]  (n wrong = 21)
  mistakes > 0.9: 3/21  AUROC 0.809  ECE 0.128  NLL 0.151

[confidence] wrote /content/drive/MyDrive/csg520_outputs/preds/a1_efficientnet_b0_test_probs.npz

=== V3 / a1_efficientnet_b0 ===  n=8146  macro-F1=0.9963 (metrics/a1_efficientnet_b0.json: 0.9967)
  confidence  correct 0.867  wrong 0.602  gap 0.264 [0.149, 0.385]  (n wrong = 18)
  mistakes > 0.9: 3/18  AUROC 0.814  ECE 0.132  NLL 0.155

[confidence] wrote /content/drive/MyDrive/csg520_outputs/preds/a1_resnet50_scratch_test_probs.npz

=== V3 / a1_resnet50_scratch ===  n=8146  macro-F1=0.9792 (metrics/a1_resnet50_scratch.json: 0.9792)
  confidence  correct 0.884  wrong 0.542  gap 0.341 [0.305, 0.375]  (n wrong = 109)
  mistakes > 0.9: 4/109  AUROC 0.934  ECE 0.108  NLL 0.154

**A4. V2 stability.** 760 test images × 16 conditions per arm. Takes a few minutes per arm and resumes after a disconnect.

In [9]:
assert HAS_GPU
!python robustness.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit {ARGS} 2>&1 | tee -a "{OUT_DIR}/robustness_gpu_log.txt"

[robustness] 760 test images (20/class), 15 perturbed conditions + clean
[a1_resnet50] clean            acc=0.997  (2s)
[a1_resnet50] brightness_0.6   acc=0.996  (2s)
[a1_resnet50] brightness_0.8   acc=0.999  (2s)
[a1_resnet50] brightness_1.2   acc=0.997  (3s)
[a1_resnet50] brightness_1.4   acc=0.997  (3s)
[a1_resnet50] contrast_0.6     acc=0.997  (3s)
[a1_resnet50] contrast_0.8     acc=0.997  (3s)
[a1_resnet50] contrast_1.2     acc=0.999  (3s)
[a1_resnet50] contrast_1.4     acc=0.996  (3s)
[a1_resnet50] rotation_-30     acc=0.997  (3s)
[a1_resnet50] rotation_-15     acc=0.999  (3s)
[a1_resnet50] rotation_15      acc=0.999  (3s)
[a1_resnet50] rotation_30      acc=0.993  (3s)
[a1_resnet50] blur_1           acc=0.980  (2s)
[a1_resnet50] blur_2           acc=0.820  (3s)
[a1_resnet50] blur_3           acc=0.550  (3s)

=== V2 / a1_resnet50 ===  images=760  clean acc=0.997
  brightness  flip 0.001 [0.000, 0.003]  retention 1.000  conf drop +0.002  worst level 0.003
  contrast    flip 0.001 [

In [11]:
!python explain.py --arms a1_resnet50 a1_efficientnet_b0 a1_resnet50_scratch a3_cnn a4_vit --patch 56 --methods occlusion --ckpt-dir "{CKPT_DIR}" --output-dir "{OUT_DIR}/sens_p56" --mask-root "{MASK_ROOT}"

[explain] sample: 380 images (10/class), LIME on 114 (A2: 2/class) | fill=mean mean RGB=(110.5, 120.0, 93.1)
[explain] mask index: 54306 segmented images
[a1_resnet50] 10/380 images  (0.1s/img, ~1 min left)
[a1_resnet50] 20/380 images  (0.1s/img, ~0 min left)
[a1_resnet50] 30/380 images  (0.1s/img, ~0 min left)
[a1_resnet50] 40/380 images  (0.1s/img, ~0 min left)
[a1_resnet50] 50/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 60/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 70/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 80/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 90/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 100/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 110/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 120/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 130/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 140/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 150/380 images  (0.0s/img, ~0 min left)
[a1_resnet50] 160/380 images  (0.0s/img, ~0 m

# B. Runtime B (CPU): A2 random forest
The RF file is about 1.2 GB, so each step spends about a minute loading it.

**B1. V3 confidence for A2**, from the cached test features (the exact rows behind the reported 0.881).

In [ ]:
!python confidence.py --arms a2_rf --n-jobs -1 {ARGS} 2>&1 | tee -a "{OUT_DIR}/confidence_cpu_log.txt"

**B2. Accuracy check before the long A2 V1 run.** This computes no maps, only A2's clean accuracy on the 380 V1 images, now with full-frame features.
It must come out near **0.89**. The old crop-based run gave 0.68. If the check fails, the cell stops and the next cell will not run.

In [ ]:
cmd = ["python", "explain.py", "--arms", "a2_rf", "--check-accuracy", "--n-jobs", "-1",
       "--ckpt-dir", str(CKPT_DIR), "--output-dir", str(OUT_DIR)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-3000:], res.stderr[-3000:])
found = re.search(r"clean accuracy on \d+ sample images: ([0-9.]+)", res.stdout)
assert found, "accuracy line not found: see the error above"
A2_ACC = float(found.group(1))
A2_OK = A2_ACC >= 0.80
assert A2_OK, f"A2 sample accuracy {A2_ACC:.3f} is too low (expected ~0.89): do not run B3, report this"
print(f"A2 accuracy {A2_ACC:.3f}: OK, run B3")

**B3. A2 V1 run (occlusion + LIME, full-frame features).** Long: 30–90 min depending on cores. Resumes after a disconnect (re-run B2 first; it is quick).

In [ ]:
assert globals().get("A2_OK"), "run B2 first (and it must pass)"
!python explain.py --arms a2_rf --n-jobs -1 {ARGS} --mask-root "{MASK_ROOT}" 2>&1 | tee -a "{OUT_DIR}/explain_fp32_a2_log.txt"

**B4. V2 stability for A2.** 760 images × 16 conditions, all feature extraction on CPU, roughly 30–60 min. Resumes after a disconnect.

In [ ]:
!python robustness.py --arms a2_rf --n-jobs -1 {ARGS} 2>&1 | tee -a "{OUT_DIR}/robustness_cpu_log.txt"

# C. Tables and figures (either runtime, once A and B are done)

In [ ]:
!python explain.py --summarize {ARGS}
!python robustness.py --summarize {ARGS}
!python confidence.py --summarize {ARGS}
from IPython.display import display
for name, path in [("V1", OUT_DIR / "explain" / "v1_table.csv"),
                   ("V2", OUT_DIR / "robustness" / "v2_table.csv"),
                   ("V3", OUT_DIR / "confidence" / "v3_table.csv")]:
    if path.exists():
        print(f"\n==== {name}: {path.name}"); display(pd.read_csv(path))
    else:
        print(f"\n==== {name}: not produced yet")

In [ ]:
from IPython.display import Image
for p in [OUT_DIR / "robustness" / "v2_flip_curves.png", OUT_DIR / "confidence" / "v3_confidence.png",
          *sorted((OUT_DIR / "explain").glob("a*/examples.png"))]:
    if p.exists():
        print(p.relative_to(OUT_DIR)); display(Image(str(p)))